# Figuras VQE Para Slides

Notebook experimental para gerar gráficos a partir dos caches VQE/FCI mais recentes. As figuras são salvas em `outputs/figures/slides/`.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from qiskit.primitives import StatevectorEstimator

from src.vqe.grid_search import run_vqe_grid_search
from src.vqe.molecular_system import statevector_grid_systems
from src.visualization.vqe_plots import (
    best_vqe_by_point,
    chemical_accuracy_config_table,
    ensure_figures_dir,
    load_latest_fci_curves,
    load_latest_vqe_results,
    plot_ansatz_comparison,
    plot_chemical_accuracy_rate,
    plot_dissociation_curve,
    plot_optimizer_comparison,
    plot_runtime_by_configuration,
    save_figure,
)

pd.set_option("display.max_columns", None)
output_dir = ensure_figures_dir()
seed = 137

## Carregar Dados Mais Recentes

In [ ]:
vqe_df = load_latest_vqe_results()
fci_df = load_latest_fci_curves(strategy="densest")

print(f"VQE rows: {len(vqe_df)}")
print(f"FCI/CASCI rows: {len(fci_df)}")
vqe_df

## Melhores Resultados Por Ponto

In [ ]:
best_df = best_vqe_by_point(vqe_df)
best_df[[
    "molecule",
    "basis",
    "distance",
    "ansatz",
    "reps",
    "optimizer",
    "energy",
    "reference_energy",
    "abs_error_kcal_mol",
    "within_chemical_accuracy",
    "total_experiment_s",
]].sort_values(["molecule", "basis", "distance"])

## DataFrame De Configurações

Resumo completo das combinações `statevector`. Nenhuma base é descartada: use `reached_chemical_accuracy` ou `accurate_points > 0` para filtrar só o que atingiu precisão química. A coluna `is_best` marca a melhor configuração por molécula.


In [ ]:
statevector_config_df = chemical_accuracy_config_table(vqe_df, only_accurate=False)

config_cols = [
    "molecule",
    "basis",
    "ansatz",
    "reps",
    "optimizer",
    "points",
    "accurate_points",
    "accuracy_rate",
    "reached_chemical_accuracy",
    "best_error_kcal_mol",
    "mean_error_kcal_mol",
    "median_error_kcal_mol",
    "best_distance",
    "mean_time_s",
    "rank",
    "is_best",
]

statevector_config_df[config_cols].sort_values([
    "molecule",
    "basis",
    "reached_chemical_accuracy",
    "accuracy_rate",
    "mean_error_kcal_mol",
], ascending=[True, True, False, False, True])


In [ ]:
configs_with_chemical_accuracy = statevector_config_df[
    statevector_config_df["reached_chemical_accuracy"]
].copy()

configs_with_chemical_accuracy[config_cols].sort_values([
    "molecule",
    "rank",
    "basis",
    "accuracy_rate",
], ascending=[True, True, True, False])


In [ ]:
best_statevector_configs = statevector_config_df[
    statevector_config_df["is_best"]
].copy()

best_statevector_configs[config_cols].sort_values("molecule")


## Curvas De Dissociação

In [ ]:
for molecule, basis in [
    ("H2", "sto-3g"),
    ("LiH", "sto-3g"),
    ("Li2O_linear", "6-31g"),
]:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_dissociation_curve(fci_df, vqe_df, molecule=molecule, basis=basis, ax=ax)
    path = save_figure(fig, output_dir, f"dissociacao_{molecule}_{basis}.png")
    print(path)
    plt.show()

## Comparação De Ansatz

Fixamos base e otimizador; cada linha mostra o melhor erro por distância para cada ansatz.

In [ ]:
for molecule, basis, optimizer in [
    ("H2", "sto-3g", "cobyla"),
    ("LiH", "sto-3g", "cobyla"),
    ("Li2O_linear", "6-31g", "cobyla"),
]:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_ansatz_comparison(vqe_df, molecule=molecule, basis=basis, optimizer=optimizer, ax=ax)
    path = save_figure(fig, output_dir, f"ansatz_{molecule}_{basis}_{optimizer}.png")
    print(path)
    plt.show()

## Comparação De Otimizadores

Fixamos molécula, base e ansatz; cada linha mostra o melhor erro por distância para cada otimizador.

In [ ]:
for molecule, basis, ansatz in [
    ("H2", "sto-3g", "efficient_su2"),
    ("LiH", "sto-3g", "efficient_su2"),
    ("Li2O_linear", "6-31g", "efficient_su2"),
]:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_optimizer_comparison(vqe_df, molecule=molecule, basis=basis, ansatz=ansatz, ax=ax)
    path = save_figure(fig, output_dir, f"otimizadores_{molecule}_{basis}_{ansatz}.png")
    print(path)
    plt.show()

## Precisão Química

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_chemical_accuracy_rate(vqe_df, ax=ax)
path = save_figure(fig, output_dir, "taxa_precisao_quimica.png")
print(path)
plt.show()

## Tempo De Execução

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_runtime_by_configuration(vqe_df, ax=ax)
path = save_figure(fig, output_dir, "tempo_medio_configuracao.png")
print(path)
plt.show()

## Figuras Geradas

In [ ]:
for path in sorted(output_dir.glob("*.png")):
    print(path)